# Stage 2 Geometry Filter Ablation — LightGlue (C1) and RoMa (C2)

Replaces ASpanFormer with alternative matchers on the **same DINO retrieval candidates**.  
Everything downstream (vggt_signals, pose_scoring) runs unchanged.

```
Normal:  retrieval_manifest → geometry_filter.py           → vggt_signals → pose_scoring
C1:      retrieval_manifest → geometry_filter_lightglue.py → vggt_signals → pose_scoring
C2:      retrieval_manifest → geometry_filter_roma.py      → vggt_signals → pose_scoring
```

**Before running this notebook:**
1. Run `python _local/convert_dino_output.py` locally to produce `dino_retrieval_manifest.jsonl`
2. Upload to Drive:
   - `dino_retrieval_manifest.jsonl` → `pipeline_output/retrieval_manifest.jsonl`
   - `aspan_all_manifest_shard1_reconstructed.jsonl` and `aspan_all_manifest_shard2_reconstructed.jsonl` → `pipeline_output/`
     (these define true shard membership — the full ASpanFormer pass+fail record per shard,
     ~7,181 / ~7,129 sources; **not** `match_manifest_shard*.csv`, which is a small human-reviewed
     subset used only by `eval_stage2.py` for scoring, not for shard membership)
   - `geometry_filter_lightglue.py` and `geometry_filter_roma.py` → `scripts/`
   - `vggt_signals.py` and `pose_scoring.py` → `scripts/`

**Outputs** (written to Drive as each stage completes, not just at the end):
```
ablation_outputs/
    c1_lightglue_shard1/
        lightglue_all_manifest.jsonl        <- saved right after the C1 Shard 1 cell
        vggt_candidates_manifest.jsonl
        lightglue_sidecars/*.npz
        vggt_output/
            vggt_judged_manifest.jsonl      <- saved right after the C1 VGGT Shard 1 cell
            true_matches_manifest.jsonl
            vggt_judge_summary.json
    c1_lightglue_shard2/   (same structure)
    c2_roma_shard1/        (same structure, roma_all_manifest.jsonl + roma_sidecars/)
    c2_roma_shard2/        (same structure)
    c1_lightglue_shard1.jsonl   <- flat copy of vggt_judged_manifest, for eval_stage2.py
    c1_lightglue_shard2.jsonl
    c2_roma_shard1.jsonl
    c2_roma_shard2.jsonl
```
The geometry-filter and VGGT stages sync independently, so a Colab disconnect mid-run
never loses more than the stage currently in flight -- run `cell-copy-drive` at any time
to check exactly what has landed.

In [ ]:
# ── Install dependencies ──────────────────────────────────────────────────────
# kornia for LightGlue (usually pre-installed in Colab; ensure >= 0.7.0)
!pip install -U kornia -q

# RoMa's latest pyproject.toml uses uv_build as its build backend.
# Step 1: install uv so that uv_build is importable in the current environment.
# Step 2: set CUDA_HOME + PATH so nvcc finds PyTorch's CUDA headers.
# Step 3: install with --no-build-isolation so the build sees the host PyTorch.
import os
os.environ['CUDA_HOME'] = '/usr/local/cuda'
os.environ['PATH'] = '/usr/local/cuda/bin:' + os.environ.get('PATH', '')

!pip install uv -q                      # provides uv_build build backend
!pip uninstall romatch -y -q 2>/dev/null; true
!pip install git+https://github.com/Parskatt/RoMA.git einops timm \
    --no-cache-dir --no-build-isolation

import torch, kornia
print('PyTorch :', torch.__version__)
print('kornia  :', kornia.__version__)
print('CUDA    :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU     :', torch.cuda.get_device_name(0))

try:
    from romatch import roma_outdoor
    print('romatch : OK')
except ImportError as e:
    print('romatch : FAILED —', e)

# local_corr is the fused CUDA correlation kernel. geometry_filter_roma.py now calls
# roma_outdoor(..., use_custom_corr=False), so this is NOT required -- "MISSING" below
# is expected and fine, not something to chase by fixing nvcc/CUDA_HOME.
try:
    import local_corr
    print('local_corr (CUDA ext) : OK (unused -- use_custom_corr=False)')
except ImportError:
    import subprocess
    nvcc = subprocess.run(['which', 'nvcc'], capture_output=True, text=True).stdout.strip()
    print(f'local_corr (CUDA ext) : MISSING (expected, unused -- nvcc={nvcc or "not found"})')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path

# ── Adjust these paths to match your Drive layout ────────────────────────────
DRIVE_ROOT = Path('/content/drive/MyDrive/ImageSimilarity')
LOCAL_WORK = Path('/content/work')

# Stage 1 output — complete DINO retrieval manifest (no shard split at this stage)
# Produced locally by: python _local/convert_dino_output.py --old-dir ... --output ...
RETRIEVAL_MANIFEST = DRIVE_ROOT / 'pipeline_output/retrieval_manifest.jsonl'

# Reconstructed full ASpanFormer manifests — define TRUE shard membership
# (all top-10 candidates per source, pass+fail; NOT match_manifest_shard*.csv,
# which is a small human-reviewed subset used only by eval_stage2.py)
SHARD1_RECONSTRUCTED = DRIVE_ROOT / 'pipeline_output/aspan_all_manifest_shard1_reconstructed.jsonl'
SHARD2_RECONSTRUCTED = DRIVE_ROOT / 'pipeline_output/aspan_all_manifest_shard2_reconstructed.jsonl'

# Image archives (same zips used in retrieval_ablation_colab)
SOURCE_ZIP = DRIVE_ROOT / 'workingsourcecrops.zip'
TARGET_ZIP = DRIVE_ROOT / 'working_targetcrops.zip'
LOCAL_SOURCE_DIR = Path('/content/source')
LOCAL_TARGET_DIR = Path('/content/target')

# VGGT checkpoint
VGGT_CKPT = DRIVE_ROOT / 'weights/VGGT-Omega/vggt_omega_1b_512.pt'

# Where scripts live on Drive
SCRIPTS_DIR = DRIVE_ROOT / 'scripts'

# Drive output folder
DRIVE_OUT = DRIVE_ROOT / 'ablation_outputs'

# Number of pairs the smoke-test cell processes (must match --max-pairs there)
SMOKE_MAX_PAIRS = 10

# Local working dirs for geometry filter + VGGT outputs
C1_OUT_S1 = LOCAL_WORK / 'c1_lightglue/shard1'
C1_OUT_S2 = LOCAL_WORK / 'c1_lightglue/shard2'
C2_OUT_S1 = LOCAL_WORK / 'c2_roma/shard1'
C2_OUT_S2 = LOCAL_WORK / 'c2_roma/shard2'

for d in [C1_OUT_S1, C1_OUT_S2, C2_OUT_S1, C2_OUT_S2, DRIVE_OUT, LOCAL_WORK]:
    d.mkdir(parents=True, exist_ok=True)

import shutil

def sync_dir_to_drive(local_dir, drive_dir, filenames, sidecar_dirname=None):
    """Copy named manifest files (if present) to Drive, and incrementally sync a sidecar dir.

    Manifests are small -- overwritten in full on every call. Sidecar .npz files can
    number in the hundreds, so only files not already present at the Drive destination
    are copied, keeping repeat calls across --resume re-runs cheap.
    """
    drive_dir.mkdir(parents=True, exist_ok=True)
    for name in filenames:
        src = local_dir / name
        if src.exists():
            shutil.copy2(src, drive_dir / name)
            print(f'  Saved: {name}')
        else:
            print(f'  MISSING (not yet produced): {name}')
    if sidecar_dirname:
        src_dir = local_dir / sidecar_dirname
        if src_dir.exists():
            dst_dir = drive_dir / sidecar_dirname
            dst_dir.mkdir(parents=True, exist_ok=True)
            all_npz = list(src_dir.glob('*.npz'))
            n_new = 0
            for npz in all_npz:
                dst = dst_dir / npz.name
                if not dst.exists():
                    shutil.copy2(npz, dst)
                    n_new += 1
            print(f'  Sidecars synced: {n_new} new ({len(all_npz)} total)')
        else:
            print(f'  MISSING (not yet produced): {sidecar_dirname}/')

# Sanity checks
for label, p in [
    ('Retrieval manifest', RETRIEVAL_MANIFEST),
    ('Shard 1 manifest',   SHARD1_RECONSTRUCTED),
    ('Shard 2 manifest',   SHARD2_RECONSTRUCTED),
    ('VGGT checkpoint',    VGGT_CKPT),
    ('Source zip',         SOURCE_ZIP),
    ('Target zip',         TARGET_ZIP),
]:
    print(f'{label}: {"OK" if p.exists() else "NOT FOUND — update DRIVE_ROOT paths"}')

In [ ]:
import shutil, sys

sys.path.insert(0, str(LOCAL_WORK))

for script_name in ['geometry_filter_lightglue.py', 'geometry_filter_roma.py',
                    'vggt_signals.py', 'pose_scoring.py']:
    src = SCRIPTS_DIR / script_name
    dst = LOCAL_WORK / script_name
    if not src.exists():
        print(f'WARNING: {script_name} not found in {SCRIPTS_DIR} — upload it first')
        continue
    shutil.copy(src, dst)
    print(f'Copied {script_name}')

In [ ]:
# ── Split complete retrieval manifest into shards + inject Colab image paths ──
import json

def load_shard_sources(jsonl_path):
    sources = set()
    with open(jsonl_path, encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            sid = json.loads(line).get('source_id')
            if sid:
                sources.add(str(sid))
    return sources

shard1_sources = load_shard_sources(SHARD1_RECONSTRUCTED)
shard2_sources = load_shard_sources(SHARD2_RECONSTRUCTED)
print(f'Unique sources in Shard 1: {len(shard1_sources)}')
print(f'Unique sources in Shard 2: {len(shard2_sources)}')

overlap = shard1_sources & shard2_sources
if overlap:
    print(f'WARNING: {len(overlap)} source IDs appear in both shard lists — Shard 1 wins for those.')
    shard2_sources -= overlap

S1_RETRIEVAL_PATCHED = LOCAL_WORK / 'shard1_retrieval.jsonl'
S2_RETRIEVAL_PATCHED = LOCAL_WORK / 'shard2_retrieval.jsonl'

counts = [0, 0, 0]  # [shard1, shard2, excluded]
excluded_sources = set()
with open(RETRIEVAL_MANIFEST, encoding='utf-8') as f_in, \
     open(S1_RETRIEVAL_PATCHED, 'w', encoding='utf-8') as f1, \
     open(S2_RETRIEVAL_PATCHED, 'w', encoding='utf-8') as f2:
    for line in f_in:
        row = json.loads(line)
        row['source_path'] = str(LOCAL_SOURCE_DIR / f"{row['source_id']}.jpg")
        row['target_path'] = str(LOCAL_TARGET_DIR / f"{row['target_id']}.jpg")
        sid = row['source_id']
        if sid in shard1_sources:
            f1.write(json.dumps(row) + '\n'); counts[0] += 1
        elif sid in shard2_sources:
            f2.write(json.dumps(row) + '\n'); counts[1] += 1
        else:
            # Not in either shard's source list (e.g. belongs to Shard 3-6) — excluded,
            # never silently guessed into a shard.
            counts[2] += 1
            excluded_sources.add(sid)

print(f'\nShard 1: {counts[0]} rows ({len(shard1_sources)} sources)')
print(f'Shard 2: {counts[1]} rows ({len(shard2_sources)} sources)')
print(f'Excluded: {counts[2]} rows across {len(excluded_sources)} sources ' \
      f'— not in either shard\'s source list (expected: Shard 3-6 sources)')

In [ ]:
# -- Selective image extraction for the smoke test -----------------------------
# Avoids the full ~45k-file unzip below: pulls only the handful of source/target
# images the smoke test's first SMOKE_MAX_PAIRS candidates actually need.
import zipfile, json as _json

def extract_specific_images(zip_path, wanted_names, dest_dir):
    dest_dir.mkdir(parents=True, exist_ok=True)
    already = {p.name for p in dest_dir.glob('*.jpg')}
    missing = wanted_names - already
    if not missing:
        print(f'{dest_dir.name}: all {len(wanted_names)} needed image(s) already present')
        return
    with zipfile.ZipFile(zip_path) as zf:
        by_basename = {}
        for name in zf.namelist():
            if '__MACOSX' in name or name.endswith('/'):
                continue
            by_basename.setdefault(Path(name).name, name)
        found = 0
        for fname in missing:
            internal = by_basename.get(fname)
            if internal is None:
                print(f'  [warn] {fname} not found in {zip_path.name}')
                continue
            with zf.open(internal) as src, open(dest_dir / fname, 'wb') as dst:
                dst.write(src.read())
            found += 1
    print(f'{dest_dir.name}: extracted {found} new image(s) '
          f'(of {len(missing)} missing, {len(wanted_names)} needed total)')

def collect_smoke_filenames(retrieval_jsonl_path, max_pairs):
    sources, targets = set(), set()
    with open(retrieval_jsonl_path, encoding='utf-8') as f:
        for i, line in enumerate(f):
            if i >= max_pairs:
                break
            row = _json.loads(line)
            sources.add(f"{row['source_id']}.jpg")
            targets.add(f"{row['target_id']}.jpg")
    return sources, targets

smoke_sources, smoke_targets = collect_smoke_filenames(S1_RETRIEVAL_PATCHED, SMOKE_MAX_PAIRS)
print(f'Smoke test needs {len(smoke_sources)} source image(s), {len(smoke_targets)} target image(s)')
extract_specific_images(SOURCE_ZIP, smoke_sources, LOCAL_SOURCE_DIR)
extract_specific_images(TARGET_ZIP, smoke_targets, LOCAL_TARGET_DIR)

## Smoke test (recommended -- run before the full shards)

The cell above (`cell-smoke-extract-images`) pulls only the handful of images this
smoke test needs directly out of the zip files, so you can run this without waiting
for the full ~45k-image unzip (that happens later, right before the C1/Shard-1 cell,
once you're ready to commit to a full run).

Runs LightGlue and RoMa on SMOKE_MAX_PAIRS pairs each. Verify:
- No import errors
- Rows appear in `vggt_candidates_manifest.jsonl`
- RoMa downloads weights without error

Remove `--max-pairs` arguments (or bump SMOKE_MAX_PAIRS) once the smoke test passes.

In [ ]:
import importlib

# --- LightGlue smoke (10 pairs) ---
import geometry_filter_lightglue as gfl
importlib.reload(gfl)
SMOKE_LG = LOCAL_WORK / 'smoke_c1'
gfl.main([
    '--input-manifest',   str(S1_RETRIEVAL_PATCHED),
    '--output-dir',       str(SMOKE_LG),
    '--breakpoint-value', '50',
    '--long_dim',         '1024',
    '--n-keypoints',      '2048',
    '--max-pairs',        str(SMOKE_MAX_PAIRS),
])

import json
cands = list((SMOKE_LG / 'vggt_candidates_manifest.jsonl').open())
print(f'LightGlue smoke: {len(cands)} pairs passed geometry filter (of {SMOKE_MAX_PAIRS} checked)')

# --- RoMa smoke (10 pairs) ---
import geometry_filter_roma as roma
importlib.reload(roma)
SMOKE_ROMA = LOCAL_WORK / 'smoke_c2'
roma.main([
    '--input-manifest',   str(S1_RETRIEVAL_PATCHED),
    '--output-dir',       str(SMOKE_ROMA),
    '--breakpoint-value', '50',
    '--n-matches',        '5000',
    '--max-pairs',        str(SMOKE_MAX_PAIRS),
])

cands_r = list((SMOKE_ROMA / 'vggt_candidates_manifest.jsonl').open())
print(f'RoMa smoke: {len(cands_r)} pairs passed geometry filter (of {SMOKE_MAX_PAIRS} checked)')

In [ ]:
# -- Full unzip of the image corpora to fast local storage --------------------
# Run this once you're ready to move past the smoke test to a full shard run.
# (The smoke test above only needed a handful of images, extracted selectively by
# cell-smoke-extract-images -- this cell pulls everything else.)
import zipfile, os, shutil

for zip_path, local_dir in [(SOURCE_ZIP, LOCAL_SOURCE_DIR), (TARGET_ZIP, LOCAL_TARGET_DIR)]:
    local_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path) as zf:
        expected = sum(
            1 for n in zf.namelist()
            if n.lower().endswith('.jpg') and '__MACOSX' not in n
        )
    n_existing = len(list(local_dir.glob('*.jpg')))
    if n_existing >= expected:
        print(f'{local_dir.name}: already has all {n_existing} .jpg files -- skipping unzip')
        continue
    print(f'{local_dir.name}: has {n_existing}/{expected} .jpg files '
          f'(likely just the smoke-test subset) -- extracting the rest ...')
    print(f'Extracting {zip_path.name} -> {local_dir} ...')
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(local_dir)
    # Flatten any subdirectory nesting (mirrors retrieval_ablation_colab.ipynb)
    for root, dirs, files in os.walk(str(local_dir)):
        if root == str(local_dir):
            continue
        for fname in files:
            src_p = Path(root) / fname
            dst_p = local_dir / fname
            if not dst_p.exists():
                shutil.move(str(src_p), str(dst_p))
    for root, _, _ in os.walk(str(local_dir), topdown=False):
        if root != str(local_dir) and not os.listdir(root):
            os.rmdir(root)

print(f'Source images: {len(list(LOCAL_SOURCE_DIR.glob("*.jpg")))} files')
print(f'Target images: {len(list(LOCAL_TARGET_DIR.glob("*.jpg")))} files')

---
## C1 — LightGlue (DISK + LightGlue via kornia)

In [ ]:
import importlib
import geometry_filter_lightglue as gfl
importlib.reload(gfl)

print('=== C1 LightGlue — Shard 1 ===')
gfl.main([
    '--input-manifest',   str(S1_RETRIEVAL_PATCHED),
    '--output-dir',       str(C1_OUT_S1),
    '--breakpoint-value', '50',
    '--long_dim',         '1024',
    '--n-keypoints',      '2048',
    '--resume',
])

print('Syncing Stage 2 (LightGlue) output to Drive ...')
sync_dir_to_drive(
    C1_OUT_S1, DRIVE_OUT / 'c1_lightglue_shard1',
    ['lightglue_all_manifest.jsonl', 'vggt_candidates_manifest.jsonl'],
    sidecar_dirname='lightglue_sidecars',
)

In [ ]:
print('=== C1 LightGlue — Shard 2 ===')
gfl.main([
    '--input-manifest',   str(S2_RETRIEVAL_PATCHED),
    '--output-dir',       str(C1_OUT_S2),
    '--breakpoint-value', '50',
    '--long_dim',         '1024',
    '--n-keypoints',      '2048',
    '--resume',
])

print('Syncing Stage 2 (LightGlue) output to Drive ...')
sync_dir_to_drive(
    C1_OUT_S2, DRIVE_OUT / 'c1_lightglue_shard2',
    ['lightglue_all_manifest.jsonl', 'vggt_candidates_manifest.jsonl'],
    sidecar_dirname='lightglue_sidecars',
)

In [ ]:
import vggt_signals, shutil
importlib.reload(vggt_signals)

print('=== C1 VGGT — Shard 1 ===')
vggt_signals.main([
    '--input-manifest', str(C1_OUT_S1 / 'vggt_candidates_manifest.jsonl'),
    '--output-dir',     str(C1_OUT_S1 / 'vggt_output'),
    '--checkpoint',     str(VGGT_CKPT),
    '--resume',
])

print('Syncing VGGT output to Drive ...')
sync_dir_to_drive(
    C1_OUT_S1 / 'vggt_output', DRIVE_OUT / 'c1_lightglue_shard1' / 'vggt_output',
    ['vggt_judged_manifest.jsonl', 'true_matches_manifest.jsonl', 'vggt_judge_summary.json'],
)

# Flat-file copy kept for eval_stage2.py compatibility (expects this exact path)
_src = C1_OUT_S1 / 'vggt_output' / 'vggt_judged_manifest.jsonl'
_dst = DRIVE_OUT / 'c1_lightglue_shard1.jsonl'
DRIVE_OUT.mkdir(parents=True, exist_ok=True)
shutil.copy(_src, _dst)
print(f'Saved to Drive: {_dst.name}  ({sum(1 for l in open(_dst) if l.strip())} rows)')

In [ ]:
print('=== C1 VGGT — Shard 2 ===')
vggt_signals.main([
    '--input-manifest', str(C1_OUT_S2 / 'vggt_candidates_manifest.jsonl'),
    '--output-dir',     str(C1_OUT_S2 / 'vggt_output'),
    '--checkpoint',     str(VGGT_CKPT),
    '--resume',
])

print('Syncing VGGT output to Drive ...')
sync_dir_to_drive(
    C1_OUT_S2 / 'vggt_output', DRIVE_OUT / 'c1_lightglue_shard2' / 'vggt_output',
    ['vggt_judged_manifest.jsonl', 'true_matches_manifest.jsonl', 'vggt_judge_summary.json'],
)

# Flat-file copy kept for eval_stage2.py compatibility (expects this exact path)
_src = C1_OUT_S2 / 'vggt_output' / 'vggt_judged_manifest.jsonl'
_dst = DRIVE_OUT / 'c1_lightglue_shard2.jsonl'
shutil.copy(_src, _dst)
print(f'Saved to Drive: {_dst.name}  ({sum(1 for l in open(_dst) if l.strip())} rows)')

---
## C2 — RoMa outdoor (dense warp-based matching)

In [ ]:
import geometry_filter_roma as roma
importlib.reload(roma)

print('=== C2 RoMa — Shard 1 ===')
roma.main([
    '--input-manifest',   str(S1_RETRIEVAL_PATCHED),
    '--output-dir',       str(C2_OUT_S1),
    '--breakpoint-value', '50',
    '--long-dim',         '1024',
    '--n-matches',        '5000',
    '--resume',
])

print('Syncing Stage 2 (RoMa) output to Drive ...')
sync_dir_to_drive(
    C2_OUT_S1, DRIVE_OUT / 'c2_roma_shard1',
    ['roma_all_manifest.jsonl', 'vggt_candidates_manifest.jsonl'],
    sidecar_dirname='roma_sidecars',
)

In [ ]:
print('=== C2 RoMa — Shard 2 ===')
roma.main([
    '--input-manifest',   str(S2_RETRIEVAL_PATCHED),
    '--output-dir',       str(C2_OUT_S2),
    '--breakpoint-value', '50',
    '--long-dim',         '1024',
    '--n-matches',        '5000',
    '--resume',
])

print('Syncing Stage 2 (RoMa) output to Drive ...')
sync_dir_to_drive(
    C2_OUT_S2, DRIVE_OUT / 'c2_roma_shard2',
    ['roma_all_manifest.jsonl', 'vggt_candidates_manifest.jsonl'],
    sidecar_dirname='roma_sidecars',
)

In [ ]:
print('=== C2 VGGT — Shard 1 ===')
vggt_signals.main([
    '--input-manifest', str(C2_OUT_S1 / 'vggt_candidates_manifest.jsonl'),
    '--output-dir',     str(C2_OUT_S1 / 'vggt_output'),
    '--checkpoint',     str(VGGT_CKPT),
    '--resume',
])

print('Syncing VGGT output to Drive ...')
sync_dir_to_drive(
    C2_OUT_S1 / 'vggt_output', DRIVE_OUT / 'c2_roma_shard1' / 'vggt_output',
    ['vggt_judged_manifest.jsonl', 'true_matches_manifest.jsonl', 'vggt_judge_summary.json'],
)

# Flat-file copy kept for eval_stage2.py compatibility (expects this exact path)
_src = C2_OUT_S1 / 'vggt_output' / 'vggt_judged_manifest.jsonl'
_dst = DRIVE_OUT / 'c2_roma_shard1.jsonl'
shutil.copy(_src, _dst)
print(f'Saved to Drive: {_dst.name}  ({sum(1 for l in open(_dst) if l.strip())} rows)')

In [ ]:
print('=== C2 VGGT — Shard 2 ===')
vggt_signals.main([
    '--input-manifest', str(C2_OUT_S2 / 'vggt_candidates_manifest.jsonl'),
    '--output-dir',     str(C2_OUT_S2 / 'vggt_output'),
    '--checkpoint',     str(VGGT_CKPT),
    '--resume',
])

print('Syncing VGGT output to Drive ...')
sync_dir_to_drive(
    C2_OUT_S2 / 'vggt_output', DRIVE_OUT / 'c2_roma_shard2' / 'vggt_output',
    ['vggt_judged_manifest.jsonl', 'true_matches_manifest.jsonl', 'vggt_judge_summary.json'],
)

# Flat-file copy kept for eval_stage2.py compatibility (expects this exact path)
_src = C2_OUT_S2 / 'vggt_output' / 'vggt_judged_manifest.jsonl'
_dst = DRIVE_OUT / 'c2_roma_shard2.jsonl'
shutil.copy(_src, _dst)
print(f'Saved to Drive: {_dst.name}  ({sum(1 for l in open(_dst) if l.strip())} rows)')

---
## Verify Drive outputs

Each geometry-filter cell (C1/C2, both shards) now saves its all-pairs manifest,
candidates manifest, and sidecars to Drive immediately on completion -- independent
of whether VGGT has run yet, so a Colab disconnect mid-pipeline doesn't lose that
stage's work. Each VGGT cell then additionally saves its judged manifest, true-match
manifest, and summary.
Run this cell at any point to check which outputs have landed.

In [ ]:
combos = [
    ('c1_lightglue_shard1', 'lightglue_all_manifest.jsonl', 'lightglue_sidecars'),
    ('c1_lightglue_shard2', 'lightglue_all_manifest.jsonl', 'lightglue_sidecars'),
    ('c2_roma_shard1',      'roma_all_manifest.jsonl',      'roma_sidecars'),
    ('c2_roma_shard2',      'roma_all_manifest.jsonl',      'roma_sidecars'),
]

def _rows(p):
    return sum(1 for l in open(p) if l.strip()) if p.exists() else None

def _status(label, n, unit='rows'):
    val = f'OK  {n} {unit}' if n is not None else '--  not yet saved'
    print(f'  {label}: {val}')

print('── Drive output status ──────────────────────────────────────────')
for combo_name, all_manifest_name, sidecar_dirname in combos:
    combo_dir = DRIVE_OUT / combo_name
    print(f'{combo_name}:')

    _status('vggt_judged (flat, eval_stage2.py input)', _rows(DRIVE_OUT / f'{combo_name}.jsonl'))
    _status(all_manifest_name, _rows(combo_dir / all_manifest_name))
    _status('vggt_candidates_manifest.jsonl', _rows(combo_dir / 'vggt_candidates_manifest.jsonl'))

    sidecar_dir = combo_dir / sidecar_dirname
    n_npz = len(list(sidecar_dir.glob('*.npz'))) if sidecar_dir.exists() else None
    _status(f'{sidecar_dirname}/', n_npz, unit='npz files')

    vggt_out = combo_dir / 'vggt_output'
    for fname in ['vggt_judged_manifest.jsonl', 'true_matches_manifest.jsonl', 'vggt_judge_summary.json']:
        n = _rows(vggt_out / fname) if fname.endswith('.jsonl') else (1 if (vggt_out / fname).exists() else None)
        _status(f'vggt_output/{fname}', n)
    print()

In [ ]:
import json

def summarise(label, out_dir):
    all_m  = out_dir / 'lightglue_all_manifest.jsonl'
    if not all_m.exists():
        all_m = out_dir / 'roma_all_manifest.jsonl'
    cand_m = out_dir / 'vggt_candidates_manifest.jsonl'
    vggt_m = out_dir / 'vggt_output' / 'vggt_judged_manifest.jsonl'

    n_all  = sum(1 for l in open(all_m)  if l.strip()) if all_m.exists()  else '?'
    n_cand = sum(1 for l in open(cand_m) if l.strip()) if cand_m.exists() else '?'
    n_vggt = sum(1 for l in open(vggt_m) if l.strip()) if vggt_m.exists() else '?'
    print(f'{label:30s}  all={n_all}  geometry_pass={n_cand}  vggt_judged={n_vggt}')

print('\n── Summary ───────────────────────────────────────────────────────')
summarise('C1 LightGlue  Shard1', C1_OUT_S1)
summarise('C1 LightGlue  Shard2', C1_OUT_S2)
summarise('C2 RoMa       Shard1', C2_OUT_S1)
summarise('C2 RoMa       Shard2', C2_OUT_S2)

print()
print('Next steps (local machine):')
print('  1. Copy c1_lightglue_shard*.jsonl and c2_roma_shard*.jsonl from Drive locally')
print('  2. python _local/eval_stage2.py --c1-shard1 ... --c2-shard1 ... (see script --help)')